In [1]:
import pandas as pd
import random
import re

class SimpleParaphraser:
    """Create paraphrased versions using synonym replacement and structure changes"""
    
    def __init__(self):
        # Simple synonym mappings for common scam phrases
        self.synonyms = {
            'urgent': ['immediate', 'pressing', 'critical', 'time-sensitive'],
            'immediately': ['right away', 'at once', 'instantly', 'now'],
            'work from home': ['remote work', 'work remotely', 'home-based position', 'telecommute'],
            'no experience': ['no background needed', 'experience not required', 'beginners welcome', 'training provided'],
            'send': ['transfer', 'wire', 'submit', 'forward'],
            'fee': ['charge', 'payment', 'cost', 'expense'],
            'contact': ['reach out', 'get in touch', 'message', 'connect with'],
            'apply': ['submit application', 'register', 'sign up', 'join'],
            'position': ['role', 'job', 'opportunity', 'opening'],
            'earn': ['make', 'receive', 'get paid', 'collect']
        }
    
    def replace_with_synonym(self, text):
        """Replace words with synonyms"""
        if pd.isna(text) or text == '':
            return text
        
        paraphrased = text
        
        for word, synonyms in self.synonyms.items():
            if word in text.lower():
                synonym = random.choice(synonyms)
                # Case-insensitive replacement
                pattern = re.compile(re.escape(word), re.IGNORECASE)
                paraphrased = pattern.sub(synonym, paraphrased, count=1)
        
        return paraphrased
    
    def paraphrase_row(self, row):
        """Paraphrase key text fields in a job posting"""
        paraphrased = row.copy()
        
        # Paraphrase text fields
        paraphrased['title'] = self.replace_with_synonym(row['title'])
        paraphrased['description'] = self.replace_with_synonym(row['description'])
        paraphrased['requirements'] = self.replace_with_synonym(row['requirements'])
        paraphrased['benefits'] = self.replace_with_synonym(row['benefits'])
        
        # Recreate full_text
        paraphrased['full_text'] = (
            str(paraphrased['title']) + ' ' +
            str(paraphrased.get('location', '')) + ' ' +
            str(paraphrased.get('department', '')) + ' ' +
            str(paraphrased.get('company_profile', '')) + ' ' +
            str(paraphrased['description']) + ' ' +
            str(paraphrased['requirements']) + ' ' +
            str(paraphrased['benefits'])
        )
        
        return paraphrased

# Test
paraphraser = SimpleParaphraser()

original = "URGENT: Work from home position. No experience needed. Send $299 fee to apply."
paraphrased = paraphraser.replace_with_synonym(original)

print("Original:")
print(original)
print("\nParaphrased:")
print(paraphrased)

Original:
URGENT: Work from home position. No experience needed. Send $299 fee to apply.

Paraphrased:
immediate: work remotely role. no background needed needed. forward $299 payment to join.


In [2]:
# Load original training data
train_df = pd.read_csv('C:/Users/diyad/Desktop/HireSafe/data/train.csv')

# Get only fraudulent postings from original EMSCAD (not synthetic)
fraud_only = train_df[train_df['fraudulent'] == 1].copy()

print(f"Found {len(fraud_only)} fraudulent postings in original training data")
print(f"We'll create 2 paraphrased versions of each = {len(fraud_only) * 2} new samples")

Found 693 fraudulent postings in original training data
We'll create 2 paraphrased versions of each = 1386 new samples


In [3]:
import numpy as np

print("Generating paraphrased versions...")

paraphrased_list = []

for idx, row in fraud_only.iterrows():
    # Create 2 paraphrased versions of each fraudulent posting
    for version in range(2):
        paraphrased = paraphraser.paraphrase_row(row)
        paraphrased_list.append(paraphrased)
    
    if idx % 100 == 0:
        print(f"Processed {idx}/{len(fraud_only)} postings...")

paraphrased_df = pd.DataFrame(paraphrased_list)

print(f"\n✓ Generated {len(paraphrased_df)} paraphrased fraudulent postings")

Generating paraphrased versions...
Processed 700/693 postings...
Processed 1000/693 postings...
Processed 2100/693 postings...
Processed 3800/693 postings...
Processed 9400/693 postings...
Processed 9600/693 postings...
Processed 11900/693 postings...
Processed 12400/693 postings...

✓ Generated 1386 paraphrased fraudulent postings


In [4]:
# Compare original vs paraphrased
print("="*80)
print("COMPARISON: Original vs Paraphrased")
print("="*80)

sample_idx = fraud_only.index[0]
original = fraud_only.loc[sample_idx]

print("\nORIGINAL:")
print(f"Title: {original['title']}")
print(f"Description: {original['description'][:200]}...")

print("\n" + "-"*80)

# Get paraphrased versions of this same posting
paraphrased_versions = paraphrased_df.head(2)

for i, (_, para) in enumerate(paraphrased_versions.iterrows(), 1):
    print(f"\nPARAPHRASED VERSION {i}:")
    print(f"Title: {para['title']}")
    print(f"Description: {para['description'][:200]}...")
    print("-"*80)

COMPARISON: Original vs Paraphrased

ORIGINAL:
Title: Customer Service Rep
Description: We are seeking a Customer Service Representative who enjoys building relationships, helping others, solving problems, and organizing a demanding schedule. The ideal candidate is independent, confident...

--------------------------------------------------------------------------------

PARAPHRASED VERSION 1:
Title: Customer Service Rep
Description: We are seeking a Customer Service Representative who enjoys building relationships, helping others, solving problems, and organizing a demanding schedule. The ideal candidate is independent, confident...
--------------------------------------------------------------------------------

PARAPHRASED VERSION 2:
Title: Customer Service Rep
Description: We are seeking a Customer Service Representative who enjoys building relationships, helping others, solving problems, and organizing a demanding schedule. The ideal candidate is independent, confident...
-------

In [5]:
# Load the synthetic scams we created earlier
train_augmented = pd.read_csv('C:/Users/diyad/Desktop/HireSafe/data/train_augmented.csv')

print(f"Current augmented data: {len(train_augmented)} samples")
print(f"Adding paraphrased data: {len(paraphrased_df)} samples")

# Combine: original + synthetic + paraphrased
final_augmented = pd.concat([train_augmented, paraphrased_df], ignore_index=True)

print(f"\nFinal augmented dataset: {len(final_augmented)} samples")

print(f"\nFinal class distribution:")
fraud_count = final_augmented['fraudulent'].sum()
legit_count = (1 - final_augmented['fraudulent']).sum()
print(f"Fraudulent: {fraud_count} ({final_augmented['fraudulent'].mean()*100:.1f}%)")
print(f"Legitimate: {legit_count} ({(1-final_augmented['fraudulent']).mean()*100:.1f}%)")

Current augmented data: 15104 samples
Adding paraphrased data: 1386 samples

Final augmented dataset: 16490 samples

Final class distribution:
Fraudulent: 2879 (17.5%)
Legitimate: 13611 (82.5%)


In [7]:
# Save final version
final_augmented.to_csv('C:/Users/diyad/Desktop/HireSafe/data/train_augmented_final.csv', index=False)

print("✓ Saved train_augmented_final.csv")

print(f"\n📊 FINAL DATA SUMMARY:")
print(f"Original EMSCAD train: ~14,304 samples (7% fraud)")
print(f"+ Synthetic scams: 800 samples")
print(f"+ Paraphrased frauds: {len(paraphrased_df)} samples")
print(f"= Final augmented: {len(final_augmented)} samples ({final_augmented['fraudulent'].mean()*100:.1f}% fraud)")

✓ Saved train_augmented_final.csv

📊 FINAL DATA SUMMARY:
Original EMSCAD train: ~14,304 samples (7% fraud)
+ Synthetic scams: 800 samples
+ Paraphrased frauds: 1386 samples
= Final augmented: 16490 samples (17.5% fraud)
